In [0]:
%run ../config/config

In [0]:
BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
BRONZE_CURRENT = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_CURRENT_TABLE}"

PRIMARY_KEY = "code"
INGESTION_COLUMN = "ingested_at"

HASH_EXCLUDE_COLUMNS = [
    INGESTION_COLUMN,
    "payload_hash"
]

spark.conf.set("spark.databricks.execution.timeout", "18000")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
if not spark.catalog.tableExists(BRONZE_CURRENT):
    window = Window.partitionBy("code").orderBy(
        F.col("ingested_at").desc()
    )

    initial_snapshot = (
        spark.table(BRONZE)
        .withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    (
        initial_snapshot.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_CURRENT)
    )

In [0]:
raw_bronze = spark.table(BRONZE)

latest_ingested_at = (
    raw_bronze
    .agg(F.max(INGESTION_COLUMN).alias("max_ingested_at"))
    .first()["max_ingested_at"]
)

if latest_ingested_at is None:
    raise ValueError("Raw Bronze table is empty.")

latest_snapshot = (
    raw_bronze
    .filter(F.col(INGESTION_COLUMN) == latest_ingested_at)
)

window = (
    Window
    .partitionBy(PRIMARY_KEY)
    .orderBy(
        F.col("last_modified_datetime").desc_nulls_last()
    )
)

latest_snapshot = (
    latest_snapshot
    .filter(F.col(PRIMARY_KEY).isNotNull())
    .withColumn("_row_number", F.row_number().over(window))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

if latest_snapshot.limit(1).count() == 0:
    raise ValueError("Latest snapshot is empty. Aborting")

hash_columns = [
    column
    for column in latest_snapshot.columns
    if column not in HASH_EXCLUDE_COLUMNS
]

latest_snapshot = latest_snapshot.withColumn(
    "payload_hash",
    F.sha2(
        F.to_json(
            F.struct(
                *[F.col(column) for column in hash_columns]
            ),
            options={"ignoreNullFields": "false"}
        ),
        256
    )
)

current_columns = spark.table(
    BRONZE_CURRENT
).columns

if "payload_hash" not in current_columns:
    spark.sql(f"""
        ALTER TABLE {BRONZE_CURRENT}
        ADD COLUMNS (payload_hash STRING)
    """)

version_before = (
    DeltaTable.forName(spark, BRONZE_CURRENT)
    .history(1)
    .select("version")
    .first()["version"]
)

latest_snapshot.createOrReplaceTempView(
    "latest_products_snapshot"
)

merge_sql = f"""
MERGE INTO {BRONZE_CURRENT} AS target

USING latest_products_snapshot AS source

ON target.{PRIMARY_KEY} = source.{PRIMARY_KEY}

WHEN MATCHED
    AND (
        target.payload_hash IS NULL
        OR target.payload_hash <> source.payload_hash
    )
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *

WHEN NOT MATCHED BY SOURCE
THEN DELETE
"""

spark.sql(merge_sql)

version_after = (
    DeltaTable.forName(spark, BRONZE_CURRENT)
    .history(1)
    .select("version")
    .first()["version"]
)

if version_after > version_before:
    cdf_df = (
        spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", version_before + 1)
        .option("endingVersion", version_after)
        .table(BRONZE_CURRENT)
    )

    display(
        cdf_df.select(
            PRIMARY_KEY,
            "_change_type",
            "_commit_version",
            "_commit_timestamp"
        )
    )

else:
    print("No changes detected, CDF is empty")

current_count = spark.table(
    BRONZE_CURRENT
).count()

print(f"Current products: {current_count}")
print(f"Processed snapshot: {latest_ingested_at}")
print(f"Processed Delta version: {version_after}")